In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_E.xpt"
csv_file = DATA_DIR / "DEMO_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 DEMO_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIDEXMON",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDAGEEX",
    "RIDRETH1",
    "DMQMILIT",
    "DMDBORN2",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDSCHOL",
    "DMDMARTL",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "INDHHIN2",
    "INDFMIN2",
    "INDFMPIR",
    "RIDEXPRG",
    "DMDHRGND",
    "DMDHRAGE",
    "DMDHRBR2",
    "DMDHREDU",
    "DMDHRMAR",
    "DMDHSEDU",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANG",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA"
]


if len(df) != 10149:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 43:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DEMO_E columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: DEMO_E structure = "
    "10,149 rows x 43 columns."
)


# ============================================================
# 4. DEFINE DECIMAL VS INTEGER FIELDS
# ============================================================

#
# IMPORTANT:
#
# These three fields must remain float / decimal:
#
# INDFMPIR = income-to-poverty ratio
# WTINT2YR = interview weight
# WTMEC2YR = MEC examination weight
#
# DO NOT convert these to Int64.
# DO NOT round them.
# DO NOT use float_format.
#

DECIMAL_COLUMNS = [
    "INDFMPIR",
    "WTINT2YR",
    "WTMEC2YR"
]


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


print("\nDecimal fields preserved AS IS:")
print(DECIMAL_COLUMNS)

print(
    "Whole-number identifier/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. EXPECTED VALIDATED TINY COUNTS
# ============================================================

EXPECTED_TINY_COUNTS = {

    "RIDAGEYR": 483,

    "WTMEC2YR": 387,

    "DMDEDUC3": 229,

    "INDFMPIR": 59,

    "RIDAGEMN": 36,

    "RIDAGEEX": 11
}


EXPECTED_TINY_TOTAL = 1205


# ============================================================
# 7. VALIDATE TINY LOCATIONS
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)


expected_tiny_columns = set(
    EXPECTED_TINY_COUNTS.keys()
)


unexpected_tiny_columns = (
    actual_tiny_columns
    -
    expected_tiny_columns
)


if unexpected_tiny_columns:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "DEMO_E variables:\n"
        + str(
            sorted(
                unexpected_tiny_columns
            )
        )
    )


missing_expected_columns = (
    expected_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected_columns:

    raise ValueError(
        "STOP: Expected tiny variables missing:\n"
        + str(
            sorted(
                missing_expected_columns
            )
        )
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: Exactly 1,205 validated "
    "tiny XPORT artifacts found."
)


# ============================================================
# 8. CHECK WTMEC2YR ZERO LOGIC
# ============================================================

#
# All 387 WTMEC2YR tiny values should belong
# to RIDSTATR = 1, Interviewed Only.
#
# CDC:
#
# RIDSTATR = 1 -> 387
# RIDSTATR = 2 -> 9,762
#

wtmec_tiny_mask = (
    df["WTMEC2YR"]
    .eq(TINY_VALUE)
)


if int(
    wtmec_tiny_mask.sum()
) != 387:

    raise ValueError(
        "Unexpected WTMEC2YR tiny count."
    )


if not (
    df.loc[
        wtmec_tiny_mask,
        "RIDSTATR"
    ]
    .eq(1)
    .all()
):

    raise ValueError(
        "STOP: WTMEC2YR tiny values occur "
        "outside RIDSTATR=1 participants."
    )


if int(
    df["RIDSTATR"]
    .eq(1)
    .sum()
) != 387:

    raise ValueError(
        "Unexpected RIDSTATR=1 count."
    )


if int(
    df["RIDSTATR"]
    .eq(2)
    .sum()
) != 9762:

    raise ValueError(
        "Unexpected RIDSTATR=2 count."
    )


print(
    "PASS: All 387 WTMEC2YR zero artifacts "
    "belong to Interviewed-Only participants."
)


# ============================================================
# 9. RESTORE ONLY VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 1,205 validated zeros restored."
)


# ============================================================
# 10. VERIFY ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{zero_count:,}"
    )


    if zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {zero_count:,}."
        )


print(
    "PASS: Restored zero counts are exact."
)


# ============================================================
# 11. VERIFY NO TINY VALUE REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 12. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn = df["SEQN"]


if not np.all(
    np.isclose(
        seqn.dropna(),
        np.round(
            seqn.dropna()
        ),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    seqn.isna()
    .sum()
)


seqn_duplicates = int(
    seqn.duplicated()
    .sum()
)


seqn_min = seqn.min()
seqn_max = seqn.max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:

    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 13. CHECK DECIMAL / FRACTIONAL VARIABLES
# ============================================================

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


print(
    "Fractional variables:"
)

print(
    fractional_columns
)


EXPECTED_FRACTIONAL_COLUMNS = {
    "INDFMPIR",
    "WTINT2YR",
    "WTMEC2YR"
}


if set(
    fractional_columns.keys()
) != EXPECTED_FRACTIONAL_COLUMNS:

    raise ValueError(
        "Unexpected fractional-variable set.\n"
        f"Expected: {EXPECTED_FRACTIONAL_COLUMNS}\n"
        f"Actual:   {set(fractional_columns.keys())}"
    )


if fractional_columns[
    "INDFMPIR"
] != 7901:

    raise ValueError(
        "Unexpected INDFMPIR fractional count."
    )


if fractional_columns[
    "WTINT2YR"
] != 10149:

    raise ValueError(
        "Unexpected WTINT2YR fractional count."
    )


if fractional_columns[
    "WTMEC2YR"
] != 9762:

    raise ValueError(
        "Unexpected WTMEC2YR fractional count."
    )


print(
    "PASS: Only INDFMPIR, WTINT2YR and "
    "WTMEC2YR contain genuine decimals."
)


# ============================================================
# 14. VERIFY INTEGER FIELDS ARE REALLY WHOLE NUMBERS
# ============================================================

unexpected_fractional_integer_fields = {}


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        unexpected_fractional_integer_fields[
            col
        ] = count


if unexpected_fractional_integer_fields:

    raise ValueError(
        "STOP: Integer-designated fields contain "
        "genuine fractional values:\n"
        + str(
            unexpected_fractional_integer_fields
        )
    )


print(
    "PASS: All non-decimal fields are whole-number values."
)


# ============================================================
# 15. CDC CHECK - SDDSRVYR
# ============================================================

#
# 5 = NHANES 2007-2008
# Count = 10,149
#

if not (
    df["SDDSRVYR"]
    .eq(5)
    .all()
):

    raise ValueError(
        "SDDSRVYR differs from expected value 5."
    )


print(
    "\nPASS: SDDSRVYR = 5 for all 10,149 records."
)


# ============================================================
# 16. CDC CHECK - RIDSTATR
# ============================================================

EXPECTED_RIDSTATR = {
    1: 387,
    2: 9762
}


actual = (
    df["RIDSTATR"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_RIDSTATR:

    raise ValueError(
        "RIDSTATR frequencies differ.\n"
        f"Expected: {EXPECTED_RIDSTATR}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: RIDSTATR matches CDC."
)


# ============================================================
# 17. CDC CHECK - RIDEXMON
# ============================================================

EXPECTED_RIDEXMON = {
    1: 4507,
    2: 5255
}


actual = (
    df["RIDEXMON"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_RIDEXMON:

    raise ValueError(
        "RIDEXMON frequencies differ."
    )


if int(
    df["RIDEXMON"]
    .isna()
    .sum()
) != 387:

    raise ValueError(
        "RIDEXMON missing count differs."
    )


print(
    "PASS: RIDEXMON matches CDC."
)


# ============================================================
# 18. CDC CHECK - RIAGENDR
# ============================================================

EXPECTED_RIAGENDR = {
    1: 5096,
    2: 5053
}


actual = (
    df["RIAGENDR"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_RIAGENDR:

    raise ValueError(
        "RIAGENDR frequencies differ."
    )


print(
    "PASS: RIAGENDR matches CDC."
)


# ============================================================
# 19. CDC CHECK - RIDAGEYR
# ============================================================

#
# CDC:
#
# 0-79 = 9,710
# 80   = 439
#

s = df["RIDAGEYR"]


if not (
    int(
        s.between(
            0,
            79
        ).sum()
    ) == 9710

    and

    int(
        s.eq(80)
        .sum()
    ) == 439

    and

    int(
        s.isna()
        .sum()
    ) == 0
):

    raise ValueError(
        "RIDAGEYR differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 483:

    raise ValueError(
        "RIDAGEYR restored zero count differs."
    )


print(
    "PASS: RIDAGEYR matches CDC/source."
)


# ============================================================
# 20. CDC CHECK - RIDAGEMN
# ============================================================

#
# CDC:
#
# 0-959 = 9,710
# Missing = 439
#

s = df["RIDAGEMN"]


if not (
    int(
        s.between(
            0,
            959
        ).sum()
    ) == 9710

    and

    int(
        s.isna()
        .sum()
    ) == 439
):

    raise ValueError(
        "RIDAGEMN differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 36:

    raise ValueError(
        "RIDAGEMN zero count differs."
    )


print(
    "PASS: RIDAGEMN matches CDC/source."
)


# ============================================================
# 21. CDC CHECK - RIDAGEEX
# ============================================================

#
# CDC:
#
# 0-959 = 9,367
# Missing = 782
#

s = df["RIDAGEEX"]


if not (
    int(
        s.between(
            0,
            959
        ).sum()
    ) == 9367

    and

    int(
        s.isna()
        .sum()
    ) == 782
):

    raise ValueError(
        "RIDAGEEX differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 11:

    raise ValueError(
        "RIDAGEEX zero count differs."
    )


print(
    "PASS: RIDAGEEX matches CDC/source."
)


# ============================================================
# 22. CDC CHECK - DMDEDUC3
# ============================================================

#
# CDC specifically reports:
#
# 0 = Never attended / Kindergarten only = 229
# Total valid = 2,500
# Missing = 7,649
#

s = df["DMDEDUC3"]


if int(
    s.eq(0)
    .sum()
) != 229:

    raise ValueError(
        "DMDEDUC3 zero count differs."
    )


if int(
    s.notna()
    .sum()
) != 2500:

    raise ValueError(
        "DMDEDUC3 valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 7649:

    raise ValueError(
        "DMDEDUC3 missing count differs."
    )


allowed_dmdeduc3 = {
    0, 1, 2, 3, 4, 5, 6, 7,
    8, 9, 10, 11, 12, 13,
    14, 15, 55, 66, 77, 99
}


actual_values = set(
    s.dropna()
    .astype(int)
    .unique()
)


if not actual_values.issubset(
    allowed_dmdeduc3
):

    raise ValueError(
        "Unexpected DMDEDUC3 code(s):\n"
        + str(
            sorted(
                actual_values
                -
                allowed_dmdeduc3
            )
        )
    )


print(
    "PASS: DMDEDUC3 matches CDC/source."
)


# ============================================================
# 23. CDC CHECK - INDFMPIR
# ============================================================

#
# CDC:
#
# 0 to 4.99 = 7,977
# 5          = 1,278
# Missing    = 894
#
# Source contains exactly 59 restored zeros.
#

s = df["INDFMPIR"]


if not (
    int(
        (
            (s >= 0)
            &
            (s < 5)
        ).sum()
    ) == 7977

    and

    int(
        s.eq(5)
        .sum()
    ) == 1278

    and

    int(
        s.isna()
        .sum()
    ) == 894
):

    raise ValueError(
        "INDFMPIR differs from CDC."
    )


if int(
    s.eq(0)
    .sum()
) != 59:

    raise ValueError(
        "INDFMPIR restored zero count differs."
    )


print(
    "PASS: INDFMPIR matches CDC/source."
)

print(
    "      Restored zeros = 59"
)


# ============================================================
# 24. CDC CHECK - WTINT2YR
# ============================================================

s = df["WTINT2YR"]


if int(
    s.notna()
    .sum()
) != 10149:

    raise ValueError(
        "WTINT2YR valid count differs."
    )


if not (
    np.isclose(
        s.min(),
        2359.373828,
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "Unexpected WTINT2YR minimum."
    )


print(
    "PASS: WTINT2YR source range validated."
)


# ============================================================
# 25. CDC CHECK - WTMEC2YR
# ============================================================

s = df["WTMEC2YR"]


if int(
    s.notna()
    .sum()
) != 10149:

    raise ValueError(
        "WTMEC2YR valid count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 387:

    raise ValueError(
        "WTMEC2YR restored zero count differs."
    )


if not np.all(
    s >= 0
):

    raise ValueError(
        "WTMEC2YR contains negative values."
    )


print(
    "PASS: WTMEC2YR matches CDC/source."
)

print(
    "      Zero MEC weights = 387"
)


# ============================================================
# 26. BUILD EXPECTED DATASET
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY the six validated tiny -> zero corrections.
#
# Do NOT:
#
# - round INDFMPIR
# - round sample weights
# - recalculate weights
# - change decimal representation
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 27. CLEAN WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# SEQN:
# 41475.0 -> 41475
#
# Codes:
# 1.0 -> 1
#
# DO NOT convert:
#
# INDFMPIR
# WTINT2YR
# WTMEC2YR
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and whole-number fields "
    "converted to clean integer formatting."
)

print(
    "PASS: INDFMPIR, WTINT2YR, WTMEC2YR "
    "left as source floats."
)


# ============================================================
# 28. PRE-EXPORT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy()
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Values match expected XPT after "
    "ONLY validated zero restorations."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO round()
# NO recalculation
#
# Missing SAS values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 31. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 32. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nTotal numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 43 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 33. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 34. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 10149:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 35. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# Integer fields:
#
# 41475.0 -> "41475"
#
#
# Decimal fields:
#
# preserve exactly as Python decoded XPT:
#
# INDFMPIR:
# 1.23 -> "1.23"
# 5.0  -> "5.0"
# 0.0  -> "0.0"
#
# WTINT2YR:
# preserve str(float(source_value))
#
# WTMEC2YR:
# preserve str(float(source_value))
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in DECIMAL_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "DEMO_E raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV representation matches "
    "expected source formatting."
)


# ============================================================
# 36. UNWANTED .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer-field '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        "Examples:",
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 remains in integer fields."
    )


print(
    "PASS: No unwanted .0 in SEQN/code fields."
)


# ============================================================
# 37. LEGITIMATE DECIMAL .0 CHECK
# ============================================================

#
# Decimal fields are allowed to end in ".0".
#
# In this source after correction:
#
# INDFMPIR = 1,354 values ending .0
# WTMEC2YR = 387 values ending .0
# WTINT2YR = 0 values ending .0
#

EXPECTED_DECIMAL_DOT_ZERO = {
    "INDFMPIR": 1354,
    "WTINT2YR": 0,
    "WTMEC2YR": 387
}


print(
    "\n--- LEGITIMATE DECIMAL .0 CHECK ---"
)


for col, expected_count in EXPECTED_DECIMAL_DOT_ZERO.items():

    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: legitimate decimal .0 "
            f"count changed."
        )


print(
    "PASS: Legitimate decimal .0 values preserved."
)


# ============================================================
# 38. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 39. FINAL TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 40. FINAL ZERO CHECK
# ============================================================

print(
    "\n--- FINAL CSV ZERO CHECK ---"
)


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    print(
        f"{col:<10}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: final zero count changed."
        )


print(
    "PASS: All final restored zero counts remain correct."
)


# ============================================================
# 41. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DEMO_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


print(
    "\nValidated tiny -> zero:"
)

for col, count in EXPECTED_TINY_COUNTS.items():

    print(
        f"{col:<10}: {count:,}"
    )


print(
    "TOTAL:",
    EXPECTED_TINY_TOTAL
)


print(
    "\nDecimal fields preserved AS IS:"
)

for col in DECIMAL_COLUMNS:

    print(col)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted integer .0 values:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "DEMO_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 DEMO_E XPT -> CSV
Rows: 10,149
Columns: 43
PASS: DEMO_E structure = 10,149 rows x 43 columns.

Decimal fields preserved AS IS:
['INDFMPIR', 'WTINT2YR', 'WTMEC2YR']
Whole-number identifier/code fields: 40

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,205

Tiny values by variable:
RIDAGEYR    483
WTMEC2YR    387
DMDEDUC3    229
INDFMPIR     59
RIDAGEMN     36
RIDAGEEX     11

PASS: Exactly 1,205 validated tiny XPORT artifacts found.
PASS: All 387 WTMEC2YR zero artifacts belong to Interviewed-Only participants.

--- ZERO RESTORATION ---
RIDAGEYR  : 483 tiny -> 0
WTMEC2YR  : 387 tiny -> 0
DMDEDUC3  : 229 tiny -> 0
INDFMPIR  : 59 tiny -> 0
RIDAGEMN  : 36 tiny -> 0
RIDAGEEX  : 11 tiny -> 0
PASS: All 1,205 validated zeros restored.

--- RESTORED ZERO COUNTS ---
RIDAGEYR  : 483
WTMEC2YR  : 387
DMDEDUC3  : 229
INDFMPIR  : 59
RIDAGEMN  : 36
RIDAGEEX  : 11
PASS: Restored zero counts are exact.

Remaining tiny artifacts: 0
PASS: No tiny artifa